# Vauban Training v3 — High Performance (Free T4)

**Upgrades over v2:**
- YOLOv8s (11M params vs 3M nano) — fits free T4
- Training at 960px (big upgrade from 640)
- Full fine-tune (no frozen layers)
- Augmented oversampling for rare classes
- Copy-paste augmentation + mixup
- 80 epochs
- Test-Time Augmentation at eval

**Requires:** Free Colab T4 (15GB VRAM)

Output: `vauban_detect_v3.pt`

In [ ]:
!pip install -q ultralytics kagglehub scikit-learn matplotlib seaborn pyyaml

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## Download & Prepare Dataset

In [ ]:
import os
os.environ['KAGGLE_API_KEY'] = 'KGAT_1fa689529f5ea2d60f8dc273b79f9ac4'

import kagglehub
mar20_path = kagglehub.dataset_download('khlaifiabilel/military-aircraft-recognition-dataset')
print(f'Dataset: {mar20_path}')

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
import shutil, random, yaml

SRC = Path(mar20_path)
OUT = Path('datasets/mar20_v3')

CLASS_MAP = {
    'A1': 'Su-35', 'A2': 'C-130', 'A3': 'C-17', 'A4': 'C-5',
    'A5': 'F-16', 'A6': 'Tu-160', 'A7': 'E-3', 'A8': 'B-52',
    'A9': 'P-3C', 'A10': 'B-1B', 'A11': 'E-8', 'A12': 'Tu-22M',
    'A13': 'F-15', 'A14': 'KC-135', 'A15': 'F-22', 'A16': 'FA-18',
    'A17': 'Tu-95', 'A18': 'KC-10', 'A19': 'Su-34', 'A20': 'Su-24',
}

CLASSES = list(dict.fromkeys(CLASS_MAP.values()))
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
print(f'{len(CLASSES)} classes: {CLASSES}')

In [ ]:
# Clean any previous run
import shutil as _sh
if OUT.exists():
    _sh.rmtree(OUT)
    print('Cleaned previous dataset')

def parse_voc_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find('size')
    w, h = int(size.find('width').text), int(size.find('height').text)
    objects = []
    for obj in root.findall('object'):
        bbox = obj.find('bndbox')
        objects.append({
            'class': obj.find('name').text,
            'xmin': int(bbox.find('xmin').text), 'ymin': int(bbox.find('ymin').text),
            'xmax': int(bbox.find('xmax').text), 'ymax': int(bbox.find('ymax').text),
            'img_w': w, 'img_h': h,
        })
    return objects

def voc_to_yolo(obj):
    mapped = CLASS_MAP.get(obj['class'])
    if not mapped or obj['img_w'] == 0 or obj['img_h'] == 0:
        return None
    cid = CLASS_TO_ID[mapped]
    w, h = obj['img_w'], obj['img_h']
    cx = max(0, min(1, ((obj['xmin'] + obj['xmax']) / 2) / w))
    cy = max(0, min(1, ((obj['ymin'] + obj['ymax']) / 2) / h))
    bw = max(0, min(1, (obj['xmax'] - obj['xmin']) / w))
    bh = max(0, min(1, (obj['ymax'] - obj['ymin']) / h))
    return f'{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}'

xml_dir = SRC / 'Annotations' / 'Horizontal Bounding Boxes'
img_dir = SRC / 'JPEGImages'
train_ids = set((SRC / 'ImageSets' / 'Main' / 'train.txt').read_text().strip().split('\n'))
test_list = list(set((SRC / 'ImageSets' / 'Main' / 'test.txt').read_text().strip().split('\n')))
random.seed(42)
random.shuffle(test_list)
mid = len(test_list) // 2
val_ids, test_ids = set(test_list[:mid]), set(test_list[mid:])

splits = {'train': train_ids, 'val': val_ids, 'test': test_ids}

for split_name, ids in splits.items():
    img_out = OUT / 'images' / split_name
    lbl_out = OUT / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    count = 0
    for img_id in sorted(ids):
        xml_path = xml_dir / f'{img_id}.xml'
        img_path = img_dir / f'{img_id}.jpg'
        if not xml_path.exists() or not img_path.exists():
            continue
        objects = parse_voc_xml(xml_path)
        lines = []
        for o in objects:
            line = voc_to_yolo(o)
            if line:
                lines.append(line)
        if not lines:
            continue
        shutil.copy2(img_path, img_out / f'{img_id}.jpg')
        (lbl_out / f'{img_id}.txt').write_text('\n'.join(lines) + '\n')
        count += 1
    print(f'{split_name}: {count} images')

data_yaml = {
    'path': str(OUT.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {i: c for i, c in enumerate(CLASSES)},
}
with open(OUT / 'data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print('\nDataset ready.')

## Augmented Oversampling for Rare Classes

In [ ]:
# Augmented oversampling for rare classes
from collections import Counter
from PIL import Image, ImageEnhance, ImageFilter

train_lbl_dir = OUT / 'labels' / 'train'
train_img_dir = OUT / 'images' / 'train'

# Count ONLY original training images (skip any _aug files from previous runs)
class_instance_counts = Counter()
file_classes = {}

for lbl_file in train_lbl_dir.glob('*.txt'):
    if '_aug' in lbl_file.stem:
        continue
    classes_in_file = set()
    for line in lbl_file.read_text().strip().split('\n'):
        if not line.strip(): continue
        cls_id = int(line.split()[0])
        class_instance_counts[CLASSES[cls_id]] += 1
        classes_in_file.add(cls_id)
    file_classes[lbl_file.stem] = classes_in_file

print(f'Original training images: {len(file_classes)}')
print(f'\nClass counts in training set:')
for c, n in sorted(class_instance_counts.items(), key=lambda x: -x[1]):
    print(f'  {c:<12} {n}')

# Target: bring rare classes up to 400 instances
target = 400

def augment_image(img_path, variation_id):
    """Create a visually different version of the image."""
    img = Image.open(img_path).convert('RGB')
    random.seed(variation_id)
    
    rot = random.choice([0, 90, 180, 270])
    if rot: img = img.rotate(rot, expand=True)
    
    h_flip = random.random() > 0.5
    v_flip = random.random() > 0.5
    if h_flip: img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if v_flip: img = img.transpose(Image.FLIP_TOP_BOTTOM)
    
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
    img = ImageEnhance.Color(img).enhance(random.uniform(0.5, 1.5))
    img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.5, 2.0))
    
    if random.random() > 0.7:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.0)))
    
    return img, rot, h_flip, v_flip

def transform_label(line, rot, h_flip, v_flip):
    parts = line.strip().split()
    if len(parts) < 5: return None
    cls_id, cx, cy, bw, bh = parts[0], float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
    
    if rot == 90:   cx, cy, bw, bh = 1 - cy, cx, bh, bw
    elif rot == 180: cx, cy = 1 - cx, 1 - cy
    elif rot == 270: cx, cy, bw, bh = cy, 1 - cx, bh, bw
    
    if h_flip: cx = 1 - cx
    if v_flip: cy = 1 - cy
    
    return f'{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}'

print(f'\nOversampling target: {target} instances per class')
print(f'Classes below target:')

copies_made = 0
for cls_idx, cls_name in enumerate(CLASSES):
    current = class_instance_counts.get(cls_name, 0)
    if current >= target or current == 0:
        continue
    
    repeats = max(1, int(target / current) - 1)
    files_with_class = [f for f, classes in file_classes.items() if cls_idx in classes]
    
    if not files_with_class:
        continue
    
    print(f'  {cls_name}: {current} -> ~{current * (repeats + 1)} ({repeats}x aug, {len(files_with_class)} source imgs)')
    
    for rep in range(repeats):
        for fname in files_with_class:
            new_name = f'{fname}_aug{rep}'
            src_img = train_img_dir / f'{fname}.jpg'
            src_lbl = train_lbl_dir / f'{fname}.txt'
            dst_img = train_img_dir / f'{new_name}.jpg'
            dst_lbl = train_lbl_dir / f'{new_name}.txt'
            
            if not src_img.exists() or dst_img.exists():
                continue
            
            variation_id = hash(f'{fname}_{rep}') % 10000
            aug_img, rot, h_flip, v_flip = augment_image(src_img, variation_id)
            aug_img.save(dst_img, quality=92)
            
            orig_lines = src_lbl.read_text().strip().split('\n')
            new_lines = [l for l in (transform_label(line, rot, h_flip, v_flip) for line in orig_lines) if l]
            
            if new_lines:
                dst_lbl.write_text('\n'.join(new_lines) + '\n')
                copies_made += 1

print(f'\nCreated {copies_made} augmented samples')
print(f'Total training images: {len(list(train_img_dir.glob("*.jpg")))}')

## Train YOLOv8s at 960px

In [ ]:
from ultralytics import YOLO

os.makedirs('models', exist_ok=True)
os.makedirs('metrics', exist_ok=True)

model = YOLO('yolov8s.pt')  # Small: 11.2M params (3.7x more than nano)

results = model.train(
    data='datasets/mar20_v3/data.yaml',
    epochs=80,
    batch=4,           # Fits free T4 at 960px
    imgsz=960,         # Big upgrade from 640, fits 15GB VRAM
    project='runs/detection',
    name='vauban_v3',
    
    # NO frozen layers — full fine-tune
    freeze=0,
    
    # Learning rate
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    
    # Aggressive augmentation for satellite imagery
    degrees=180.0,     # Aircraft can face any direction
    translate=0.15,
    scale=0.6,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,        # Blend images for regularization
    copy_paste=0.3,    # Copy aircraft between images
    
    # HSV augmentation
    hsv_h=0.02,
    hsv_s=0.5,
    hsv_v=0.4,
    
    # Perspective
    perspective=0.0005,
    shear=2.0,
    
    # Close mosaic late for fine-tuning
    close_mosaic=15,
    
    # Patience
    patience=20,
    
    # Save
    save=True,
    save_period=20,
)

best = Path('runs/detection/vauban_v3/weights/best.pt')
if best.exists():
    shutil.copy(best, 'models/vauban_detect_v3.pt')
    print(f'Model saved: {os.path.getsize("models/vauban_detect_v3.pt") / 1024 / 1024:.1f} MB')

## Metrics — Training Curves

In [ ]:
from IPython.display import display, Image as IPImage

run_dir = Path('runs/detection/vauban_v3')

for name in ['results.png', 'confusion_matrix_normalized.png',
             'BoxF1_curve.png', 'BoxPR_curve.png',
             'val_batch0_pred.jpg', 'val_batch1_pred.jpg']:
    src = run_dir / name
    if src.exists():
        shutil.copy(src, f'metrics/{name}')

print('=== Training Curves ===')
if (run_dir / 'results.png').exists():
    display(IPImage(filename=str(run_dir / 'results.png'), width=900))

In [ ]:
for title, name in [('F1 vs Confidence', 'BoxF1_curve.png'),
                     ('Precision-Recall', 'BoxPR_curve.png'),
                     ('Confusion Matrix', 'confusion_matrix_normalized.png')]:
    p = run_dir / name
    if p.exists():
        print(f'\n=== {title} ===')
        display(IPImage(filename=str(p), width=800))

In [ ]:
print('=== Validation Predictions ===')
for name in ['val_batch0_pred.jpg', 'val_batch1_pred.jpg']:
    if (run_dir / name).exists():
        display(IPImage(filename=str(run_dir / name), width=900))

## Test Set Evaluation (with TTA)

In [ ]:
from ultralytics import YOLO

val_model = YOLO('models/vauban_detect_v3.pt')

# Standard eval
print('=== Standard Evaluation ===')
val_results = val_model.val(data='datasets/mar20_v3/data.yaml', split='test', imgsz=960)
print(f'mAP@0.5:      {val_results.box.map50:.4f}')
print(f'mAP@0.5:0.95: {val_results.box.map:.4f}')
print(f'Precision:     {val_results.box.mp:.4f}')
print(f'Recall:        {val_results.box.mr:.4f}')

# TTA eval
print('\n=== With Test-Time Augmentation ===')
val_results_tta = val_model.val(data='datasets/mar20_v3/data.yaml', split='test', imgsz=960, augment=True)
print(f'mAP@0.5:      {val_results_tta.box.map50:.4f}')
print(f'mAP@0.5:0.95: {val_results_tta.box.map:.4f}')
print(f'Precision:     {val_results_tta.box.mp:.4f}')
print(f'Recall:        {val_results_tta.box.mr:.4f}')

print('\n=== Improvement ===')
print(f'mAP@0.5:  {val_results.box.map50:.4f} -> {val_results_tta.box.map50:.4f} ({(val_results_tta.box.map50 - val_results.box.map50)*100:+.1f}%)')

In [ ]:
print('\n=== Per-Class AP@0.5 ===')
names = val_model.names
print(f'{"Class":<12} {"AP@0.5":>8} {"AP@0.5:0.95":>12}')
print('-' * 34)
for i in range(len(val_results.box.ap50)):
    print(f'{names[i]:<12} {val_results.box.ap50[i]:>8.4f} {val_results.box.ap[i]:>12.4f}')

## Per-Class AP Bar Chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ap50 = val_results.box.ap50
class_names = [names[i] for i in range(len(ap50))]

sorted_idx = np.argsort(ap50)
sorted_names = [class_names[i] for i in sorted_idx]
sorted_ap = ap50[sorted_idx]

russian = {'Su-35', 'Su-34', 'Su-24', 'Tu-95', 'Tu-160', 'Tu-22M'}
colors = ['#e74c3c' if n in russian else '#2d7dd2' for n in sorted_names]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(range(len(sorted_names)), sorted_ap, color=colors)
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names, fontsize=10)
ax.set_xlabel('AP@0.5')
ax.set_title('Per-Class AP@0.5 — v3 (YOLOv8s @ 960px)')
ax.set_xlim(0, 1)
ax.grid(True, axis='x', alpha=0.3)

for bar, ap in zip(bars, sorted_ap):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{ap:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('metrics/per_class_ap50_v3.png', dpi=150, bbox_inches='tight')
plt.show()

## Compare v2 vs v3

In [ ]:
# If you have v2 model, compare side by side
try:
    v2 = YOLO('models/vauban_detect_v2.pt')
    v2_results = v2.val(data='datasets/mar20_v3/data.yaml', split='test', imgsz=960)
    
    print('=== v2 vs v3 Comparison ===')
    print(f'{"Metric":<18} {"v2 (nano)":>12} {"v3 (small)":>12} {"Delta":>8}')
    print('-' * 52)
    for name, v2_val, v3_val in [
        ('mAP@0.5', v2_results.box.map50, val_results.box.map50),
        ('mAP@0.5:0.95', v2_results.box.map, val_results.box.map),
        ('Precision', v2_results.box.mp, val_results.box.mp),
        ('Recall', v2_results.box.mr, val_results.box.mr),
    ]:
        delta = v3_val - v2_val
        print(f'{name:<18} {v2_val:>12.4f} {v3_val:>12.4f} {delta:>+8.4f}')
except:
    print('v2 model not available for comparison')

## Download

In [ ]:
# Zip everything
import shutil

# Save run artifacts
shutil.make_archive('vauban_v3_results', 'zip', 'runs/detection/vauban_v3')

print('=== Files ===')
for f in ['models/vauban_detect_v3.pt', 'vauban_v3_results.zip']:
    if os.path.exists(f):
        print(f'  {f} — {os.path.getsize(f)/1024/1024:.1f} MB')

try:
    from google.colab import files
    files.download('models/vauban_detect_v3.pt')
    files.download('vauban_v3_results.zip')
except:
    print('Not on Colab')